# NL2SQL Pipeline — Hybrid Selector

**Pipeline:**
1. Load test queries
2. Hybrid selector (best weights from grid search) -> top-1 database
3. Extract CREATE TABLE schema for selected database only
4. SQLCoder-7b-2 generates SQL from question + schema
5. Execute predicted and ground truth SQL -> execution accuracy

The NL2SQL model only sees the schema of the selected database.

## 0. Install dependencies

In [ ]:
!pip install -q rank-bm25 sentence-transformers==2.7.0 transformers accelerate bitsandbytes einops

## 1. Paths and imports

In [ ]:
import sys
import os
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE         = "/kaggle/input/datasets/eces99/bachelor-project-ece"
PROJECT_ROOT = BASE
DATABASE_DIR = f"{BASE}/data/spider/test_database"
TEST_PATH    = f"{BASE}/data/spider/test.json"
HYBRID_WEIGHTS_PATH  = f"{BASE}/models/best_hybrid_weights.json"
OUTPUT_PATH          = "/kaggle/working/results_hybrid_pipeline.json"

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, f"{PROJECT_ROOT}/tests")

# Verify all paths
for label, path in [("PROJECT_ROOT", PROJECT_ROOT), ("DATABASE_DIR", DATABASE_DIR),
                    ("TEST_PATH", TEST_PATH), ("HYBRID_WEIGHTS_PATH", HYBRID_WEIGHTS_PATH)]:
    status = "OK" if os.path.exists(path) else "NOT FOUND"
    print(f"  {label}: {status}")

In [ ]:
from src.selector.schema_repr import load_schemas, load_queries, Preprocessor
from src.selector.lexical     import LexicalSelector
from src.selector.statistical import TFIDFSelector
from src.selector.semantical  import SemanticSelector
from training_models.hybrid   import HybridSelector
from pipeline_utils import (
    get_create_table_schema, get_db_path,
    execution_match, clean_generated_sql,
    build_sqlcoder_prompt, make_result_record
)

print("Imports OK")

## 2. Load schemas and queries

In [ ]:
print("Loading schemas and queries...")
p                    = Preprocessor(remove_generic=True, lemmatize=True)
schemas_preprocessed = load_schemas(DATABASE_DIR, preprocessor=p)
raw_schemas          = load_schemas(DATABASE_DIR)
queries              = load_queries(TEST_PATH)

print(f"  {len(queries)} test queries")
print(f"  {len(raw_schemas)} databases")

## 3. Initialize Hybrid selector

In [ ]:
print("Initializing base selectors...")
bm25     = LexicalSelector(schemas_preprocessed, preprocessor=p, variant="okapi")
tfidf    = TFIDFSelector(schemas_preprocessed, preprocessor=p, ngram_range=(1, 2))
semantic = SemanticSelector(raw_schemas, model_name="thenlper/gte-small")

print("Loading hybrid weights...")
with open(HYBRID_WEIGHTS_PATH) as f:
    weights = tuple(json.load(f)["weights"])
print(f"  Weights: alpha={weights[0]}, beta={weights[1]}, gamma={weights[2]}")

hybrid_selector = HybridSelector(
    bm25_selector=bm25,
    tfidf_selector=tfidf,
    semantic_selector=semantic,
    weights=weights,
)
print("  Done.")

## 4. Load SQLCoder-7b-2

In [ ]:
MODEL_NAME = "defog/sqlcoder-7b-2"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True,
)
print("  Model loaded.")

## 5. Run the full pipeline

Uncomment `queries = queries[:50]` to do a quick sanity check first before running all 2147.

In [ ]:
results = []
exec_correct      = 0
db_select_correct = 0

# queries = queries[:50]  # uncomment for a quick test first

for i, q in enumerate(queries):
    if i % 50 == 0:
        print(f"  {i}/{len(queries)}  "
              f"exec_acc={exec_correct/(i+1):.3f}  "
              f"db_acc={db_select_correct/(i+1):.3f}")

    question   = q["question"]
    correct_db = q["db_id"]
    gold_sql   = q["query"]

    # Step 1: Database selection
    ranked      = hybrid_selector.rank(question, top_k=1)
    selected_db = ranked[0][0]
    db_correct  = (selected_db == correct_db)
    if db_correct:
        db_select_correct += 1

    # Step 2: Load schema for selected DB ONLY
    try:
        db_path = get_db_path(DATABASE_DIR, selected_db)
        schema  = get_create_table_schema(db_path)
    except FileNotFoundError:
        results.append(make_result_record(
            question, correct_db, selected_db, gold_sql, "", False, db_correct
        ))
        continue

    # Step 3: Build prompt and generate SQL
    prompt = build_sqlcoder_prompt(question, schema, selected_db)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=1.0,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    pred_sql = clean_generated_sql(generated)

    # Step 4: Execution accuracy -- always against the CORRECT db
    correct_db_path = get_db_path(DATABASE_DIR, correct_db)
    is_correct = execution_match(correct_db_path, pred_sql, gold_sql)
    if is_correct:
        exec_correct += 1

    results.append(make_result_record(
        question, correct_db, selected_db, gold_sql, pred_sql, is_correct, db_correct
    ))

    del inputs, output_ids
    torch.cuda.empty_cache()

n = len(queries)
print(f"\n{'='*50}")
print(f"Hybrid Pipeline -- Final Results")
print(f"{'='*50}")
print(f"Total queries       : {n}")
print(f"DB selection Top-1  : {db_select_correct/n:.3f} ({db_select_correct}/{n})")
print(f"Execution accuracy  : {exec_correct/n:.3f} ({exec_correct}/{n})")

## 6. Save results

In [ ]:
output = {
    "model":              "best_weights_hybrid",
    "nl2sql_model":       MODEL_NAME,
    "hybrid_weights":     list(weights),
    "total_queries":      n,
    "db_selection_top1":  round(db_select_correct / n, 4),
    "execution_accuracy": round(exec_correct / n, 4),
    "queries":            results,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved to {OUTPUT_PATH}")

## 7. Oracle upper bound

Same pipeline but always passes the **correct** database schema to SQLCoder.
This is the ceiling -- what execution accuracy would be if selection were perfect.
The gap between oracle and selector accuracy directly answers RQ4.

In [ ]:
oracle_correct = 0
oracle_results = []

for i, q in enumerate(queries):
    if i % 50 == 0:
        print(f"  {i}/{len(queries)}  oracle_acc={oracle_correct/(i+1):.3f}")

    question   = q["question"]
    correct_db = q["db_id"]
    gold_sql   = q["query"]

    # Always the correct DB -- no selector
    db_path = get_db_path(DATABASE_DIR, correct_db)
    schema  = get_create_table_schema(db_path)
    prompt  = build_sqlcoder_prompt(question, schema, correct_db)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated  = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    pred_sql   = clean_generated_sql(generated)
    is_correct = execution_match(db_path, pred_sql, gold_sql)
    if is_correct:
        oracle_correct += 1

    oracle_results.append({
        "question": question, "correct_db": correct_db,
        "gold_sql": gold_sql, "predicted_sql": pred_sql,
        "execution_match": is_correct,
    })

    del inputs, output_ids
    torch.cuda.empty_cache()

print(f"\nOracle execution accuracy         : {oracle_correct/n:.3f} ({oracle_correct}/{n})")
print(f"Hybrid selector execution accuracy: {exec_correct/n:.3f} ({exec_correct}/{n})")
print(f"Gap due to selection errors       : {(oracle_correct - exec_correct)/n:.3f}")

with open("/kaggle/working/results_hybrid_oracle.json", "w") as f:
    json.dump({"oracle_accuracy": oracle_correct/n, "queries": oracle_results}, f, indent=2)